<a href="https://colab.research.google.com/github/FrenyJeffrin/ML-wd-ScikitLearn/blob/main/llm_playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project: Building an LLM Playground

In [ ]:
import torch, transformers, tiktoken
print("torch", torch.__version__, "| transformers", transformers.__version__)


torch 2.11.0+cpu | transformers 5.12.1


Tokenization

word-level tokenization

In [ ]:
corpus = [ "The quick brown fox jumps over the lazy dog",
          "Tokenization converts text to numbers",
           "Large language models predict the next token"]

PAD, UNK ="[PAD]","[UNK]"

words=set()
for text in corpus:
  local_words = text.lower().split()
  for word in local_words:
    words.add(word)

vocab=[PAD, UNK] + list(words)
word2id={}
id2word={}

for i,w in enumerate(vocab):
  word2id[w]=i
  id2word[i]=w

# print(f"Vocabulary size:{len(vocab)} words")
# print("First 15 vocab entries:",vocab[:15])

def encode(texts):
  # ids=[]
  # for t in texts.lower().split():
  #   if t in word2id:
  #    ids.append(word2id[t])
  #   else:
  #     ids.append(word2id[UNK])
  # return ids
  return [word2id.get(t, word2id[UNK]) for t in texts.lower().split()]
# print(encode("fox jumps everywhere"))

def decode(ids):
  # words=[]
  # for id_ in ids:
  #   words.append(id2word[id_])
  # print(" ".join(words))
  return " ".join(id2word[id_] for id_ in ids if id_ != word2id[PAD])

# decode([8,12,1])

sample= 'The brown unicorn jumps'
ids=encode(sample)
recovered=decode(ids)

print("\nInput text:", sample)
print("Token Ids:", ids)
print("Decoded:", recovered)








Input text: The brown unicorn jumps
Token Ids: [8, 3, 1, 7]
Decoded: the brown [UNK] jumps


character-level tokenization

In [ ]:
import string

letters=list(string.ascii_lowercase+string.ascii_uppercase)+ [" "]
special=['PAD','UNK']
vocab=special+letters

char2id={}
id2char={}

char2id={ch:id for id,ch in enumerate(vocab)}
id2char={id:ch for ch,id in char2id.items()}
print(id2char)

def encode(text):
  return [char2id.get(ch, char2id['UNK']) for ch in text]
print(encode("Hi how are you"))

def decode(ids):
  return "".join(id2char[id] for id in ids if id != char2id['PAD'])

print(decode([35, 10, 54, 9, 16, 24, 54, 2, 19, 6, 54, 26, 16, 22]))

sample= 'The brown unicorn jumps'
ids=encode(sample)
recovered=decode(ids)

print("\nInput text:", sample)
print("Token Ids:", ids)
print("Decoded:", recovered)



{0: 'PAD', 1: 'UNK', 2: 'a', 3: 'b', 4: 'c', 5: 'd', 6: 'e', 7: 'f', 8: 'g', 9: 'h', 10: 'i', 11: 'j', 12: 'k', 13: 'l', 14: 'm', 15: 'n', 16: 'o', 17: 'p', 18: 'q', 19: 'r', 20: 's', 21: 't', 22: 'u', 23: 'v', 24: 'w', 25: 'x', 26: 'y', 27: 'z', 28: 'A', 29: 'B', 30: 'C', 31: 'D', 32: 'E', 33: 'F', 34: 'G', 35: 'H', 36: 'I', 37: 'J', 38: 'K', 39: 'L', 40: 'M', 41: 'N', 42: 'O', 43: 'P', 44: 'Q', 45: 'R', 46: 'S', 47: 'T', 48: 'U', 49: 'V', 50: 'W', 51: 'X', 52: 'Y', 53: 'Z', 54: ' '}
[35, 10, 54, 9, 16, 24, 54, 2, 19, 6, 54, 26, 16, 22]
Hi how are you

Input text: The brown unicorn jumps
Token Ids: [47, 9, 6, 54, 3, 19, 16, 24, 15, 54, 22, 15, 10, 4, 16, 19, 15, 54, 11, 22, 14, 17, 20]
Decoded: The brown unicorn jumps


subword-level tokenization

In [ ]:
from transformers import AutoTokenizer

bpe_tok = AutoTokenizer.from_pretrained("gpt2")

def encode(text):
  return bpe_tok.encode(text)
def decode(ids):
  return bpe_tok.decode(ids)

sample ="unbelievable tokenization powers! 🫣"
ids= encode(sample)
recovered = decode(ids)

print("\nInput text :", sample)
print("Token IDs :", ids)
print("Tokens :", bpe_tok.convert_ids_to_tokens(ids))
print("Decoded :", recovered)



Input text : unbelievable tokenization powers! 🫣
Token IDs : [403, 6667, 11203, 540, 11241, 1634, 5635, 0, 12520, 104, 96]
Tokens : ['un', 'bel', 'iev', 'able', 'Ġtoken', 'ization', 'Ġpowers', '!', 'ĠðŁ', '«', '£']
Decoded : unbelievable tokenization powers! 🫣


TikToken

In [ ]:
import tiktoken

sentence="The ⭐ star-player scored 40 points!"

encodings =[('gpt2', tiktoken.get_encoding('gpt2')),('cl100k_base', tiktoken.get_encoding('cl100k_base')) ]

for name,enc in encodings:
  print(f"\n=== {name} ===")
  print("vocabulary size: ", enc.n_vocab)

  ids = enc.encode(sentence)
  tokens= [enc.decode([i]) for i in ids]
  print(f"sentence splits into {len(ids)} tokens: ")
  print(list(zip(tokens, ids)))


=== gpt2 ===
vocabulary size:  50257
sentence splits into 11 tokens: 
[('The', 464), (' �', 2343), ('�', 255), ('�', 238), (' star', 3491), ('-', 12), ('player', 7829), (' scored', 7781), (' 40', 2319), (' points', 2173), ('!', 0)]

=== cl100k_base ===
vocabulary size:  100277
sentence splits into 10 tokens: 
[('The', 791), (' �', 2928), ('��', 99834), (' star', 6917), ('-player', 43467), (' scored', 16957), (' ', 220), ('40', 1272), (' points', 3585), ('!', 0)]


Language Model

In [ ]:
import torch.nn as nn

class Linear(nn.Module):
  def __init__(self, in_features, out_features):
    super(Linear, self).__init__()
    self.weights= nn.Parameter(torch.randn(out_features, in_features))
    self.bias = nn.Parameter(torch.randn(out_features))

  def forward(self, x):
    return torch.matmul(x, self.weights.t()) + self.bias

linear = Linear(3,4)
x = torch.randn(1,3)
y= linear(x)
print(y)


tensor([[ 2.1671,  1.5907, -1.2464,  0.5039]], grad_fn=<AddBackward0>)


In [ ]:
import torch.nn as nn, torch

lin = nn.Linear(3,2)
x = torch.tensor((1.0,-1.0,0.5))
print("Input :", x)
print("Weights :", lin.weight)
print("Bias :", lin.bias)
print("Output :", lin(x))

Input : tensor([ 1.0000, -1.0000,  0.5000])
Weights : Parameter containing:
tensor([[-0.4124, -0.4178, -0.4088],
        [ 0.4164,  0.5493, -0.4138]], requires_grad=True)
Bias : Parameter containing:
tensor([-0.0333,  0.1796], requires_grad=True)
Output : tensor([-0.2323, -0.1602], grad_fn=<ViewBackward0>)


A Transformer Layer

In [ ]:
import torch
from transformers import  GPT2LMHeadModel

#Load the 124 M-parameter GPT-2 and inspect its layers(12 layers)

gpt2= GPT2LMHeadModel.from_pretrained("gpt2")
block= gpt2.transformer.h[0]
for name ,module in block.named_children():
  print(name, module.__class__.__name__)


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
gpt2.config


NameError: name 'gpt2' is not defined

In [ ]:
seq_len=8
dummy_tokens=torch.randint(0,gpt2.config.vocab_size, (1,seq_len))
print(dummy_tokens)
with torch.no_grad():
  #embed tokens+positions the same way GPT-2 does
  #forward through one layer
  hidden= gpt2.transformer.wte(dummy_tokens)+gpt2.transformer.wpe(torch.arange(seq_len))
  out= block(hidden)[0]

print("\nOutput shape :", out.shape)


LLM's output

In [ ]:
import torch, torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

#Load gpt2 model and tokenizer
try:
  gpt2
except NameError:
  gpt2= GPT2LMHeadModel.from_pretrained("gpt2")
  tokenizer= GPT2TokenizerFast.from_pretrained("gpt2")

#Tokenize input text
text= "Hello my name"
input_ids=tokenizer(text, return_tensors="pt").input_ids #shape: (1, seq_len)

#get logits by passing the ids to the gpt2 model.
with torch.no_grad():
  logits=gpt2(input_ids).logits
print("Logits shape :", logits.shape)

#predict next token
probs = F.softmax(logits(0,-1), dim=1)
topk= torch.topk(probs,5)

print("\ntop-5 predictions for the next token:")
for idx,p in zip(topk.indices.tolist(), topk.values.tolist()):
  print(f"{tokenizer.decode([idx]):>10s} - {p:.4f}")




Generation

Greedy decoding

In [ ]:
from transformers import AutoTokenizer, AutoModelForCasualLM
MODELS={
    "gpt2": "gpt2",
}

tokenizers, models={},{}
device = 'cuda' if torch.cuda.is_available() else "cpu"
for key, mid in MODELS.items():
  tok= Autokenizer.from_pretrained(mid)
  mdl= AutoModelForCasualLM.from_pretrained(mid).eval().to(device)
  if tok.pad_token is None:
    tok.pad_token=tok.eos_token
  mdl.config.pad_tok_id= tok.pad_tok_id
  tokenizers[key], models[key]=tok,mdl
  print(f"Loaded {mid} as {key}")

def generate(model_key,prompt,strategy="greedy",max_new_tokens=100):
  tok,mdl=tokenizers[model_key],models[model_key]
  enc=tok(prompt, return_tensors="pt").to(mdl.device)
  gen_args=dict(**enc, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id)
  if strategy=="greedy":
    gen_args["do_sample"]=False
  elif strategy=="top_k":
    gen_args.update(dict(do_sample=True, top_k=50, temperature=0.9))
  elif strategy=="top_p":
    gen_args.update(dict(do_sample=True, top_p=0.9, temperature=0.9))
  out= mdl.generate(**gen_args)
  return tok.decode(out[0], skip_special_toens=True)




In [ ]:
tests= ["Once upon a time","what is 2+2?", "Suggest a party theme."]
for prompt in tests:
  print(f"\n==GPT2 | Greedy ==")
  print(generate("gpt2", prompt, "greedy", 80))


Top-k or top_p sampling

In [ ]:
tests= ["Once upon a time","what is 2+2?", "Suggest a party theme."]
for prompt in tests:
  print(f"\n==GPT2 | Top-p ==")
  print(generate("gpt2", prompt, "top-p", 40))

Instruction-tuned LLMs

Qwen1.5-8B vs. GPT2

In [ ]:
from transformers import AutoTokenizer, AutoModelForCasualLM
MODELS={
    "gpt2": "gpt2",
    "qwen": "Qwen/Qwen3-0.6B",
    "gemma": "google/gemma-3-270m"
}

tokenizers, models={},{}
device = 'cuda' if torch.cuda.is_available() else "cpu"
for key, mid in MODELS.items():
  tok= Autokenizer.from_pretrained(mid)
  mdl= AutoModelForCasualLM.from_pretrained(mid).eval().to(device)
  if tok.pad_token is None:
    tok.pad_token=tok.eos_token
  mdl.config.pad_tok_id= tok.pad_tok_id
  tokenizers[key], models[key]=tok,mdl
  print(f"Loaded {mid} as {key}")

In [ ]:
tests=[{"Once upon a time","greedy"},{"what is 2+2?", "top-k"}, {"Suggest a party theme.", "top-p"}]
for prompt,strategy in tests:
  for key in ["qwen"]:
    print(f"\n=={key.upper()} | {strategy} ==")
    print(generate(key,prompt,strategy,80))